# Freshness

Create a policy, wait until search and the graph can see it, close its ACL,
and confirm the engineering employee loses it while the HR admin keeps it.
Then delete it and confirm both search and `find_related` drop it.

The read-time ACL check can hide the policy before the worker catches up.
The delete still waits for the worker to remove the graph node. Before
creating the same id again, the notebook waits until the raw snapshot is
gone. A version-1 row with the same ACL is otherwise skipped. The body
includes the clock time so that create is not the hash the last delete
tombstoned.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "gateway_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import gateway_demo
import hr_demo


In [ ]:
from datetime import UTC, datetime

POLICY_ID = "pol-demo-gateway"
PHRASE = f"zephyr notebook policy {datetime.now(UTC).strftime('%Y%m%d%H%M%S')}"
BODY = f"# Demo\n\n{PHRASE}\n"
PAYLOAD = {
    "policy_id": POLICY_ID,
    "title": "Zephyr notebook policy",
    "category": "leave",
    "region": "DE",
    "format": "markdown",
    "native_acl": ["hr_employee"],
    "body": BODY,
}

def seen(bearer: str) -> bool:
    found = gateway_demo.search(bearer, PHRASE)
    return any(item["object_id"] == POLICY_ID for item in found["results"])

employee = gateway_demo.token("employee_eng")
admin = gateway_demo.token("hr_admin")
key = f"raw/hr/policy/{POLICY_ID}.md"
hr_demo.delete_if_present("policy", POLICY_ID)
hr_demo.wait_until(lambda: hr_demo.snapshot(key) is None, hr_demo.DELETE_WAIT, "snapshot gone")
hr_demo.wait_until(lambda: not seen(employee) and not seen(admin), hr_demo.DELETE_WAIT, "old copy gone")
created = hr_demo.admin("POST", "/admin/policies", PAYLOAD)
assert created["policy_id"] == POLICY_ID

hr_demo.wait_until(lambda: seen(employee), hr_demo.CATCHUP_WAIT, "gateway index")
visible = gateway_demo.related(employee, "policies", entity_id=POLICY_ID)
assert any(step["id"] == POLICY_ID for path in visible["paths"] for step in path["steps"])

changed = hr_demo.admin(
    "PUT", f"/admin/policies/{POLICY_ID}/acl", {"native_acl": ["hr_policy_admin"]}
)
assert changed["native_acl"] == ["hr_policy_admin"]
hr_demo.wait_until(lambda: not seen(employee), hr_demo.WRITE_WAIT, "acl closed")
assert seen(admin)

hr_demo.delete_if_present("policy", POLICY_ID)
hr_demo.wait_until(lambda: not seen(employee) and not seen(admin), hr_demo.DELETE_WAIT, "deleted from search")

def graph_gone() -> bool:
    graph = gateway_demo.related(admin, "policies", entity_id=POLICY_ID)
    ids = [step["id"] for path in graph["paths"] for step in path["steps"]]
    return POLICY_ID not in ids

hr_demo.wait_until(graph_gone, hr_demo.DELETE_WAIT, "deleted from graph")
print("freshness ok", POLICY_ID)
